# Mittag-Leffler PDF in MWright Function

MWright Function script

In [1]:
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from healpy.newvisufunc import projview, newprojplot

from astropy.io import fits
import pandas as pd

from scipy.optimize import curve_fit
import scipy.special as scsp
from scipy.special import gammaln, gammasgn, gamma

import scipy.stats as stats
from statsmodels.graphics.tsaplots import plot_acf
from scipy.fft import fft, fftfreq
import inspect


## Expressing Gaussian-Mittag-Leffler Integral as M-Wright Infinite Series

The probability density function expressed as a Gaussian weighted by the M-Wright function:

$$
\boxed{
p(d)=
\int_0^\infty
\frac{1}{\sqrt{2\pi\zeta r}}
\exp\left(-\frac{d^2}{2\zeta r}\right)
M_\beta(r)\,dr
}
\tag{1}
$$

where $d\in\mathbb R$, $\zeta>0$, $r>0$, and $0<\beta<1$.

Fourier transform of probability distributions, the original Gaussian is (see equation A.8)

$$
\boxed{
p(d)=
\frac{1}{\sqrt{2\zeta}}
M_{\beta/2}
\left(
|d|\sqrt{\frac{2}{\zeta}}
\right).
}
\tag{2}
$$

Infinite-series representation of the M-Wright function (see equation 3.8):

$$
\boxed{
M_\nu(z)=
\sum_{n=0}^{\infty}
\frac{(-z)^n}
{n!\,\Gamma[1-\nu(n+1)]}
}
\tag{2}
$$


Note:


$$
\boxed{
\mathrm{MSD}
=
\frac{\zeta}{\Gamma(\beta+1)}.
}
$$

Consequently,

$$
\boxed{
\zeta=\mathrm{MSD}\,\Gamma(\beta+1).
}
$$

This relation connects the analytical PDF to the empirical
MSD used in the numerical analysis.


**Reference:**
Mainardi, F., Mura, A., and Pagnini, G. (2010).
   *The M-Wright Function in Time-Fractional Diffusion
   Processes: A Tutorial Survey*.
   https://doi.org/10.1155/2010/104505

In [ ]:
def M_wright(z, nu, n_terms=200, pole_tol=1e-12):
    """
    Mainardi (M-Wright) function M_nu(z) for z>=0, 0<nu<1:
      M_nu(z) = sum_{k>=0} (-z)^k / (k! * Gamma(1 - nu*(k+1)) )

    Uses log-Gamma + gamma sign for numerical stability.
    """
    z = np.asarray(z, dtype=float)
    z = np.maximum(z, 0.0)

    if not (0.0 < nu < 1.0):
        raise ValueError("nu must satisfy 0 < nu < 1.")

    out = np.zeros_like(z)
    logz = np.where(z > 0, np.log(z), -np.inf)

    for k in range(n_terms):
        a = 1.0 - nu*(k+1)  # Gamma argument: Gamma(a)

        # Skip poles at a = 0, -1, -2, ...
        if (a <= 0.0) and np.isclose(a, np.round(a), atol=pole_tol):
            continue

        # term sign: (-1)^k * sign(1/Gamma(a)) = (-1)^k * sign(Gamma(a))
        sign = ((-1)**k) * gammasgn(a)

        # |term| = exp(k log z - log(k!) - log|Gamma(a)|)
        log_mag = k*logz - gammaln(k+1) - gammaln(a)

        term = sign * np.exp(log_mag)
        term = np.where(np.isfinite(term), term, 0.0)
        out += term

    return out

def mwright_pdf_from_zeta(d, msd, nu, n_terms=200):
    """
    Symmetric two-sided M-Wright PDF on R, msd = zeta/Gamma(beta + 1);
    self similarity of MWright function p(d)

      p(d) = (1/(2 s)) * M_nu(|d|/s)
      with s = sqrt(zeta / 2) = sqrt(msd * Gamma(2nu+1)/2)

    For nu=1/2 this becomes exactly Gaussian N(mu, sigma^2).
    """
    # sigma = np.abs(sigma)
    if msd == 0:
        return np.zeros_like(x, dtype=float)

    # s = sigma * np.sqrt(gamma(2.0*nu + 1.0) / 2.0)
    s = np.sqrt(msd* gamma(2*nu + 1) / 2)
    z = np.abs(np.asarray(d)) / s
    return 0.5/s * M_wright(z, nu, n_terms=n_terms)